#  Mini Project — Student Performance Prediction

## Objective
Build a simple machine learning classification project that predicts whether a student is likely to **PASS or FAIL** using study hours, attendance, previous marks, assignment score, and project requirements.

**Model:** Logistic Regression
**Target:** Result (PASS/FAIL)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the dataset
df = pd.read_csv("student_performance_pass_fail.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nClass distribution:")
print(df["Result"].value_counts())

Dataset shape: (120, 6)


,Study_Hours,Attendance,Previous_Marks,Assignment_Score,Project_Requirements,Result
0,7.6,81.3,83.9,95.5,Met,PASS
1,4.7,84.2,81.8,42.8,Met,PASS
2,8.3,58.8,45.1,42.6,Met,PASS
3,6.9,73.7,64.5,40.7,Met,FAIL
4,1.8,56.9,69.4,77.8,Met,FAIL



Class distribution:
Result
PASS    70
FAIL    50
Name: count, dtype: int64


In [3]:
# Clean the data
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill numerical missing values with medians, if any
numeric_cols = ["Study_Hours", "Attendance", "Previous_Marks", "Assignment_Score"]
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill missing categorical values with the most frequent value
df["Project_Requirements"] = df["Project_Requirements"].fillna(
    df["Project_Requirements"].mode()[0]
)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
Study_Hours             0
Attendance              0
Previous_Marks          0
Assignment_Score        0
Project_Requirements    0
Result                  0
dtype: int64

Missing values after cleaning:
Study_Hours             0
Attendance              0
Previous_Marks          0
Assignment_Score        0
Project_Requirements    0
Result                  0
dtype: int64


In [4]:
# Select features and target
features = [
    "Study_Hours",
    "Attendance",
    "Previous_Marks",
    "Assignment_Score",
    "Project_Requirements"
]
target = "Result"

X = df[features]
y = df[target]

# Convert the categorical feature into a numeric column
X = pd.get_dummies(X, columns=["Project_Requirements"], drop_first=True, dtype=int)

print("Features after encoding:")
display(X.head())
print("Target:")
display(y.head())

Features after encoding:


,Study_Hours,Attendance,Previous_Marks,Assignment_Score,Project_Requirements_Not Met
0,7.6,81.3,83.9,95.5,0
1,4.7,84.2,81.8,42.8,0
2,8.3,58.8,45.1,42.6,0
3,6.9,73.7,64.5,40.7,0
4,1.8,56.9,69.4,77.8,0


Target:


0    PASS
1    PASS
2    PASS
3    FAIL
4    FAIL
Name: Result, dtype: object

In [5]:
# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 96
Testing samples: 24


In [6]:
# Train the Logistic Regression classifier
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [7]:
# Make predictions and calculate accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred, labels=["FAIL", "PASS"]))

Model Accuracy: 83.33%

Classification Report:
              precision    recall  f1-score   support

        FAIL       0.80      0.80      0.80        10
        PASS       0.86      0.86      0.86        14

    accuracy                           0.83        24
   macro avg       0.83      0.83      0.83        24
weighted avg       0.83      0.83      0.83        24

Confusion Matrix (rows = actual, columns = predicted):
[[ 8  2]
 [ 2 12]]


In [8]:
# Test the model with 3 new student inputs
new_students = pd.DataFrame({
    "Study_Hours": [6.0, 2.5, 8.0],
    "Attendance": [85.0, 65.0, 94.0],
    "Previous_Marks": [72.0, 42.0, 88.0],
    "Assignment_Score": [82.0, 45.0, 91.0],
    "Project_Requirements": ["Met", "Not Met", "Met"]
})

new_encoded = pd.get_dummies(
    new_students,
    columns=["Project_Requirements"],
    drop_first=True,
    dtype=int
)

new_encoded = new_encoded.reindex(columns=X.columns, fill_value=0)

predictions = model.predict(new_encoded)
probabilities = model.predict_proba(new_encoded).max(axis=1)

prediction_results = new_students.copy()
prediction_results["Prediction"] = predictions
prediction_results["Confidence"] = (probabilities * 100).round(1)

display(prediction_results)

,Study_Hours,Attendance,Previous_Marks,Assignment_Score,Project_Requirements,Prediction,Confidence
0,6.0,85.0,72.0,82.0,Met,PASS,96.4
1,2.5,65.0,42.0,45.0,Not Met,FAIL,99.9
2,8.0,94.0,88.0,91.0,Met,PASS,99.8


## Example interpretation

- A student with **6 study hours, 85% attendance, 72 previous marks, 82 assignment score, and project requirements met** is predicted as PASS.
- A student with **2.5 study hours, 65% attendance, 42 previous marks, 45 assignment score, and project requirements not met** is predicted as FAIL.
- A student with **8 study hours, 94% attendance, 88 previous marks, 91 assignment score, and project requirements met** is predicted as PASS.

The prediction is based only on the features supplied to the model and should not be treated as a guaranteed academic outcome.
